[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/80_magic_vine_solution.ipynb)

# Solution: Magic Vine Cutting

Reference solution — prefix DP over running max/min last-occurrence.

## 解析

**结论：剩余藤蔓永远是一个前缀，设 `dp[k]` 为「把长度为 `k` 的前缀全部剪掉的最少操作数」，则 `dp[k] = 1 + min(dp[maxpos[k]-1], dp[minpos[k]-1])`。**

### 状态定义
每次操作都是「找某位置，把它及右侧全剪掉」，剪完后剩下的仍是一个**前缀**。所以整个过程的状态可以只用「当前前缀长度 `k`」刻画。定义 `dp[k]` = 从长度 `k` 的前缀出发、直到剪空（长度 0）的最少操作数，`dp[0] = 0`。

### 转移
对长度为 `k` 的前缀 `a[0..k-1]`，一次操作有两种选择：
- 剪「最大值最后一次出现」的位置 `maxpos[k]`（1-based）→ 剩下前缀长度 `maxpos[k]-1`；
- 剪「最小值最后一次出现」的位置 `minpos[k]` → 剩下前缀长度 `minpos[k]-1`。

两者取更优：

```
dp[k] = 1 + min(dp[maxpos[k]-1], dp[minpos[k]-1])
```

### 预处理 maxpos / minpos
从左到右扫一遍，维护「到目前为止的运行最大值/最小值，及其最后出现下标」。注意「最后一次出现」意味着**并列时取更大的下标**，所以更新用 `>=`（最大）和 `<=`（最小）：新来的相等元素会把位置刷新到更右。

### 直觉
如果前缀的最大值或最小值就在**第 1 段**（`maxpos[k]==1` 或 `minpos[k]==1`），那么一次操作就能连第 1 段一起剪掉（跳到 `dp[0]`），`dp[k]=1`。样例 `[1,3,2]` 最小值 1 在第 1 段 → 1 次；`[5,5,5]` 最大=最小，最后一次出现永远是当前末尾，只能从右往左一段段剥 → 3 次。

### 复杂度
预处理和 DP 各一遍，`O(n)` 时间、`O(n)` 空间。

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
from typing import List

In [ ]:
# ✅ SOLUTION

class Solution:
    def min_operations(self, a: List[int]) -> int:
        n = len(a)
        maxpos = [0] * (n + 1)   # 1-based last index of running max over a[0..k-1]
        minpos = [0] * (n + 1)
        cmax = cmin = None
        mp = np_ = 0
        for k in range(1, n + 1):
            v = a[k - 1]
            if cmax is None or v >= cmax:   # >= keeps the LAST occurrence
                cmax = v; mp = k
            if cmin is None or v <= cmin:
                cmin = v; np_ = k
            maxpos[k] = mp
            minpos[k] = np_

        dp = [0] * (n + 1)
        for k in range(1, n + 1):
            dp[k] = 1 + min(dp[maxpos[k] - 1], dp[minpos[k] - 1])
        return dp[n]

In [ ]:
# Demo
sol = Solution()
print(sol.min_operations([1, 3, 2]))      # 1
print(sol.min_operations([4, 2, 4, 1]))   # 2
print(sol.min_operations([5, 5, 5]))      # 3

In [ ]:
from torch_judge import check
check('magic_vine')